# UI = f(state), server-first — the React & Next.js mental model

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 37 §37.2–§37.3 · type: concept-lab

**The promise:** by the end you can internalize *the UI is a function of state* and the Next.js **server/client component boundary** — without writing a line of React — by simulating the render-on-state-change loop in Python and reading the real `.tsx` it corresponds to.

**The medium can't run React.** A Python kernel can't execute React or Next.js, so this is a *concept-lab*: we *simulate and diagram* the render loop, one-way data flow, and the server/client split in Python, then annotate the book's real `.tsx`. The actual UI lives in [`templates/web-starter/`](../../../templates/web-starter/) and [`capstone/web/`](../../../capstone/web/) (built in Ch 38), and this notebook ends by pointing there.

Runs free and offline by default (`MOCK=1`): pure standard-library Python, no Node runtime, no API key.

## 🧠 Why this matters

React looks intimidating because tutorials drown it in syntax. The core is one equation: **the UI is a function of state.** You write functions — *components* — that take data and return a description of markup; React calls them whenever their inputs change and reconciles the result with the real page. You never imperatively poke the DOM ("find this div, change its text"); you change *state*, and the UI follows.

That reframes every frontend feature as a system-design question you already ask: *what is the minimal source-of-truth state, and where does it live?* Next.js then adds a server half — by default components run **only on the server**, ship zero JavaScript, and can hit your FastAPI service directly; only the interactive leaves opt into the browser. Get those two ideas — state-driven render, server-by-default — and you can direct, review, and own an AI frontend without memorizing React trivia. See §37.2–§37.3.

## Objectives & prereqs

**By the end you can:**
- Explain and *simulate* `UI = render(state)` plus one-way dataflow (props down, events up).
- Decide what is the minimal **source-of-truth state** and what should be **derived in render**.
- Recognize **`useEffect` abuse** (copying props→state, computing derived values, chaining updates) and why it causes extra renders, stale data, and infinite loops.
- Place the Next.js **server/client boundary** at the interactive leaves and explain why `"use client"` is *infectious*.
- Use the chapter's review checklist as the rubric for reviewing AI-generated components.

**Prereqs:** [`37-01-typescript-for-python-engineers.ipynb`](37-01-typescript-for-python-engineers.ipynb). Chapter 37 §37.2–§37.3 read. Ch 25/26 (the FastAPI backend the BFF fronts) for context.

**Packages:** standard library only. No Node/Deno, no model calls anywhere.

In [ ]:
# Setup — imports, env, the MOCK switch, a fixed seed.
import os
import random

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default): fully offline, pure-Python simulation of the React/Next.js model.
# No model calls happen anywhere in this notebook; the switch is here for consistency
# with the rest of the course (and gates the optional live exercise).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(37)  # determinism for anything stochastic

print(f"MOCK mode: {MOCK}  (offline, pure-Python simulation of React/Next.js)")
print("Node/Deno: not required — no React is executed; we simulate and read .tsx.")

## The core: UI = render(state)

A component is a *pure function* from state to markup. Change the state, re-run the function, and the page follows. Let's make that literal: a tiny `render(state) -> markup string` plus a `set_state` that re-invokes `render`. There is no "page object" we mutate — the markup is *derived* from state, every time.

This mirrors the book's `RunCard` (§37.2): a status badge whose text is *computed from* `status`, and a button that *changes* `status`.

In [ ]:
# A minimal React simulator: state in, markup out. The whole point is that nothing
# is mutated in place — every render DERIVES the UI from the current state.
def render(state: dict) -> str:
    """Pure function: UI = f(state). Mirror of the book's <RunCard> component."""
    status = state["status"]
    badge = f'<span class="badge">{status}</span>'
    button = '<button>Cancel</button>'
    return f"<div>{badge}{button}</div>"


# The 'screen' is just whatever render() last returned. set_state re-renders.
_screen = {"markup": ""}


def set_state(state: dict, **changes) -> dict:
    """Like React's setter from useState: produce NEW state, then re-render."""
    new_state = {**state, **changes}        # never mutate; derive a new state
    _screen["markup"] = render(new_state)   # React would reconcile this with the DOM
    return new_state


state = {"status": "queued"}
_screen["markup"] = render(state)
print("initial :", _screen["markup"])

state = set_state(state, status="cancelled")  # the button's onClick, in spirit
print("after click:", _screen["markup"])
print("\nThe UI changed because STATE changed — not because we edited the markup.")

## Read the real thing: `<RunCard>` line by line

Here is the book's actual React (§37.2). Annotate it against the Python simulation you just ran:

```tsx
function RunStatus({ status }: { status: string }) {
  return <span className="badge">{status}</span>;   // props DOWN (read-only arg)
}

function RunCard({ runId }: { runId: string }) {
  const [status, setStatus] = useState("queued");   // state hook: [value, setter]
  return (
    <div>
      <RunStatus status={status} />                 {/* derived in render */}
      <button onClick={() => setStatus("cancelled")}>Cancel</button>  {/* events UP */}
    </div>
  );
}
```

- `const [status, setStatus] = useState("queued")` is our `state` dict + `set_state` — one source-of-truth value and its setter.
- `<RunStatus status={status} />` is *props down*: `status` flows into a child as a read-only argument. The badge text is **derived in render**, never stored twice.
- `onClick={() => setStatus("cancelled")}` is *events up*: the child reports an event; the owner of the state changes it; the UI re-derives.
- The HTML-ish syntax is **JSX** — function calls in disguise. `<RunStatus status={x} />` compiles to a call to `RunStatus({status: x})`.
- **Hooks** are the `use*` functions that give components capabilities (`useState`, `useEffect`, `useContext`). The rule: call them only at the top level, never in conditions or loops — React tracks them *by call order*.

## 🔮 Predict: store it, or derive it?

You have a list of runs and you want to show the **count of failed runs**. Two designs:

- **(A) Store it:** keep `failed_count` in its *own* state, and update it whenever `runs` changes.
- **(B) Derive it:** compute `failed_count` from `runs` during render; store nothing extra.

Predict, *before running the next cell*: which design can show a number that *disagrees* with the list on screen? What's the name of that failure in backend terms?

In [ ]:
# (A) STORE the derived value in its own state — the anti-pattern.
stateA = {"runs": ["ok", "failed", "ok"], "failed_count": 1}  # count maintained by hand
# A new run arrives, but the code forgot to also bump failed_count (a real, common bug):
stateA = {**stateA, "runs": stateA["runs"] + ["failed"]}        # updated runs...
# ...failed_count NOT updated -> now it's stale.
print("(A) stored:  runs =", stateA["runs"])
print("             failed_count says", stateA["failed_count"],
      "but the list actually has", stateA["runs"].count("failed"), "<- DISAGREES")

# (B) DERIVE it in render — there is only one source of truth, so it can't drift.
stateB = {"runs": ["ok", "failed", "ok"]}
stateB = {**stateB, "runs": stateB["runs"] + ["failed"]}
derived_failed = stateB["runs"].count("failed")                 # computed every render
print("\n(B) derived: runs =", stateB["runs"])
print("             failed_count =", derived_failed, "<- always agrees, by construction")

**What you just saw.** Storing the count duplicated information already in `runs`, and the two drifted the instant one update forgot the other — the frontend's version of *denormalized data with no sync strategy*. Deriving the count in render keeps a single source of truth, so it cannot disagree. The architectural rule from §37.2: **state is the only thing that changes; everything computable from it is derived during render, not stored.**

## ⚠️ Pitfall: `useEffect` abuse (the most common React bug)

The single most common React bug — in human *and* AI-generated code — is `useEffect` abuse: using effects to copy props into state, to compute derived values, or to chain state updates ("when X changes, set Y"). Each causes extra renders, stale data, and infinite loops.

```tsx
// ⚠️ ANTI-PATTERN: an effect that derives state from state. This can loop forever.
useEffect(() => {
  setFailedCount(runs.filter(r => r === "failed").length);  // setting state...
}, [runs, failedCount]);  // ...whose change re-triggers this effect. Round and round.
```

The rule of thumb: **an effect is *only* for synchronizing with something *outside* React** — a network socket (Ch 25's WebSocket), a browser API, a third-party widget. If you can compute it during render, compute it during render. Let's *feel* the infinite loop by simulating the effect-runs-on-every-change cycle.

In [ ]:
# Simulate React's loop: after each render, run effects; if an effect SETS state,
# that schedules another render, which runs effects again... A correct effect reaches
# a steady state (no change -> no re-render). A buggy one never settles.
def run_loop(effect, initial, max_renders=12):
    state = dict(initial)
    for render_count in range(1, max_renders + 1):
        new_state = effect(dict(state))
        if new_state == state:
            print(f"  settled after {render_count} render(s): {state}")
            return
        state = new_state  # state changed -> React schedules ANOTHER render
    print(f"  STILL changing after {max_renders} renders -> infinite loop. last: {state}")


# BAD effect: derives failed_count via an effect, and bumps a tick every time, so the
# state never stops changing -> the render/effect cycle never settles.
def bad_effect(state):
    state["failed_count"] = state["runs"].count("failed")
    state["tick"] = state.get("tick", 0) + 1   # the classic 'effect mutates a dep' trap
    return state


# GOOD effect: only mirrors an EXTERNAL value (e.g., a socket's last message). Once it
# matches, nothing changes -> the loop settles immediately.
external_socket_value = "connected"
def good_effect(state):
    state["connection"] = external_socket_value  # sync with the OUTSIDE world only
    return state


print("BAD effect (derives state in an effect, bumps a tick):")
run_loop(bad_effect, {"runs": ["ok", "failed"]})
print("\nGOOD effect (syncs with an external socket value only):")
run_loop(good_effect, {"connection": "disconnected"})

**What you just saw.** The bad effect never settles — each render changes state, which schedules another render, forever; in a browser that's a frozen tab and a maxed-out CPU. The good effect synchronizes with an *external* value and reaches a steady state in one pass. When you review AI-generated components, **hunt the effects first**: that is where the bodies are buried. For *server* data specifically, don't hand-roll `useEffect` + `fetch` + loading flags at all — reach for TanStack Query / SWR, or skip client fetching with a server component (next).

## Next.js: React with a server half

React renders components but has no opinion about routing, data loading, or servers. **Next.js** (its modern *App Router*) supplies those, and it feels backend-shaped: routing is filesystem-based.

```text
app/
  layout.tsx            shared shell: nav, theme
  page.tsx              /
  chat/page.tsx         /chat
  runs/[id]/page.tsx    /runs/123   (dynamic segment)
  api/chat/route.ts     POST /api/chat   (route handler)
```

The big idea is the **server/client component split**. By default every component is a *server component*: it runs **only on the server**, can be `async`, can query a database or call your FastAPI service directly, and ships **zero JavaScript** — the browser receives rendered output. Only components marked `"use client"` become *client components*: hydrated in the browser, allowed state, effects, and event handlers.

```tsx
// app/runs/page.tsx — a server component: async, no hooks, no JS shipped
export default async function RunsPage() {
  const res = await fetch(`${process.env.API_URL}/runs`, {
    headers: { Authorization: `Bearer ${await serviceToken()}` },
    cache: "no-store",                  // always fresh
  });
  const runs: Run[] = await res.json();
  return <RunList runs={runs} />;
}
```

Notice what this replaces: **no `useEffect`, no loading spinner, no exposing the API URL or credentials to the browser.** The server fetched the data and sent HTML. Push interactivity to the *leaves* and keep everything above them on the server. Let's simulate that split: tag each node `server` or `client` and tally what ships.

In [ ]:
# A tiny render tree. Each node is server-by-default; only interactive leaves opt into
# the client with "use client". A server node may touch secrets (it never reaches the
# browser); a client node must not.
tree = {
    "name": "RunsPage", "client": False, "touches_secret": True,  # async server fetch
    "children": [
        {"name": "RunList", "client": False, "touches_secret": False, "children": [
            {"name": "RunRow", "client": False, "touches_secret": False, "children": [
                # the ONE interactive leaf: a Cancel button needs an onClick handler.
                {"name": "CancelButton", "client": True, "touches_secret": False, "children": []},
            ]},
        ]},
    ],
}


def walk(node, depth=0):
    tag = '"use client"' if node["client"] else "server"
    ships_js = node["client"]
    # SECURITY: a secret must never live in a client component (it ships to the browser).
    leak = node["client"] and node["touches_secret"]
    flag = "  <-- SECRET LEAK!" if leak else ""
    print("  " * depth + f"{node['name']:<14} [{tag:^12}] ships_js={ships_js}{flag}")
    js_nodes = 1 if ships_js else 0
    for child in node["children"]:
        js_nodes += walk(child, depth + 1)
    return js_nodes


shipped = walk(tree)
print(f"\nComponents shipping JavaScript to the browser: {shipped} of 4 (just the leaf).")
print("The async fetch + service token stayed on the server. No spinner, no leak.")

## ⚠️ Pitfall: `"use client"` is infectious

Everything imported *by* a client component becomes client code too. The classic failure — common in AI-generated codebases — is a stray `"use client"` near the root `layout.tsx` that silently drags the **whole app** into the browser bundle, forfeiting server components entirely. The app still works, so nobody notices; it's just slower, heavier, and leakier than it should be.

Let's make the contagion visible as a small reachability sim: mark one node `"use client"` and watch everything it imports turn client too.

In [ ]:
# Import graph: node -> modules it imports. "use client" is infectious DOWN the imports:
# a client module pulls everything it imports into the client bundle as well.
imports = {
    "layout":   ["ThemeToggle", "Nav"],
    "Nav":      ["NavLink"],
    "NavLink":  [],
    "ThemeToggle": [],
    "RunsPage": ["RunList"],
    "RunList":  ["RunRow"],
    "RunRow":   [],
}


def client_reachable(roots):
    """Everything reachable from a "use client" root is forced into the client bundle."""
    seen, stack = set(), list(roots)
    while stack:
        n = stack.pop()
        if n in seen:
            continue
        seen.add(n)
        stack.extend(imports.get(n, []))
    return seen


# CASE 1 — the directive sits on a tiny leaf (correct): only it + its imports go client.
print('Case 1: "use client" on ThemeToggle (a leaf):')
print("  forced client:", sorted(client_reachable(["ThemeToggle"])))

# CASE 2 — a stray directive lands on the root layout (the bug): it infects everything
# the layout imports, which is the whole shell.
print('\nCase 2: stray "use client" on layout (the bug):')
print("  forced client:", sorted(client_reachable(["layout"])))
print("  -> the entire shell now ships to the browser; server components are forfeited.")

**What you just saw.** A `"use client"` on a leaf stays contained; the same directive near the root pulls the whole import-graph into the browser bundle. In review, **grep for the directive and challenge each occurrence**: does this component truly need state or event handlers? If not, the directive — and the bloat it drags in — shouldn't be there.

## 🎯 Senior lens: thin BFF, thick service

The architectural question a backend engineer answers once per project: *what logic lives in Next.js's server half versus the FastAPI backend?* The pattern that scales is **backend-for-frontend (BFF)**. The Python service owns the *domain* — agents, RAG, queues, the database — and exposes the enterprise-grade API of Chapter 26. The Next.js server layer owns *presentation concerns* — session cookies, request shaping, streaming pass-through, per-page data assembly. Resist growing business logic in the BFF "because it's closer": you'd end up with two half-backends and one ownership dispute. **Thin BFF, thick service.** (This sets up Chapter 38, where the BFF streams agent events to the UI.)

## 📋 Review checklist for AI-generated components

The chapter's rubric (§37, the checklist) — run AI-written React through this before you trust it:

- **Types:** Are all API boundaries validated at runtime (Zod), not just typed for the compiler? *(see 37-01)*
- **Events:** Are streamed/variant payloads modeled as discriminated unions, narrowed in `switch`?
- **State:** Is each piece of UI state stored exactly once, with everything else derived in render?
- **Effects:** Is every `useEffect` synchronizing with an external system — and nothing else?
- **Boundary:** Are components server-by-default, with `"use client"` only at interactive leaves?
- **Secrets:** Do tokens and internal URLs stay in server components, actions, and route handlers?
- **BFF:** Is the Next.js server layer thin — presentation glue only — with the domain in FastAPI?

## Recap

- **UI = f(state):** components are pure functions from state to markup; you change *state*, the UI follows. Data flows *down* (props), events flow *up* (callbacks).
- **Store state once; derive the rest in render.** Duplicated state is denormalized data with no sync — it drifts.
- **`useEffect` is only for syncing with something *outside* React.** Using it to derive or chain state causes extra renders, stale data, and infinite loops. Hunt effects first in review.
- **Next.js is server-first:** components are server-by-default (zero JS, can touch secrets/DB/FastAPI); only `"use client"` leaves hydrate in the browser. `"use client"` is *infectious* up the import graph.
- **Thin BFF, thick service:** Next.js owns presentation; FastAPI owns the domain.

## Exercises

Each task *changes* something; predict the outcome before you run it.

1. **Derive vs store.** Extend the predict cell with a `running_count` and a `done_count`. Implement both the (A) stored and (B) derived versions, then drop one run and predict which counts go stale.
2. **Fix the loop.** Change `bad_effect` so the render/effect loop *settles*: remove the `tick` bump and move the `failed_count` computation out of the effect (derive it in render instead). Predict how many renders it now takes to settle.
3. **Place the boundary.** Add a `RunFilter` dropdown (needs `onChange`) to the render-tree sim. Predict whether it should be `"use client"`, mark it, and re-run `walk` to confirm only it ships JS.
4. **Catch a leak.** In the render-tree sim, set `CancelButton["touches_secret"] = True` and re-run. Predict the warning, then state the fix (move the secret to a server action / route handler).
5. **(Optional, live)** No API key needed for this notebook. If you want, sketch a `route.ts`-style handler in Python (`httpx`) that proxies one FastAPI call — the BFF pattern in miniature.

In [ ]:
# Exercise scratch space — your code here.


In [ ]:
# Exercise scratch space — your code here.


## Next

- **Book:** §37.2 (React mental model) and §37.3 (Next.js server/client split); §37.4 covers Tailwind + shadcn/ui, the styling system the capstone uses.
- **Where this leads:** Chapter 38 (*Building AI Interfaces*) builds the streaming UI that renders the `AgentEvent` union from [`37-01`](37-01-typescript-for-python-engineers.ipynb) — the BFF streams, the client narrows and shows the agent thinking.
- **The real code:** now open [`templates/web-starter/`](../../../templates/web-starter/) — the Next.js App Router starter (TypeScript · Tailwind · shadcn/ui) you copy into a job — and the complete reference frontend in [`capstone/web/`](../../../capstone/web/). You built the mental model and the review rubric here; that's where you apply both.